<a id="pendulum-equation"></a>

$$
\text{单摆 (Simple Pendulum): 长度 L 的轻杆, 末端质量 m, 偏离竖直方向的角度 } \theta \\ \quad \\
\text{推导 1 (牛顿第二定律, 切向分量):} \\
\text{弧长 } s = L\theta \implies \text{切向加速度 } \frac{d^2 s}{dt^2} = L \ddot{\theta} \\
\text{重力 mg 沿切向的分量 } -mg\sin\theta \text{ (指向平衡位置), 绳的拉力沿径向, 不贡献切向分量} \\
m L \ddot{\theta} = -mg \sin\theta \implies \ddot{\theta} = -\frac{g}{L}\sin\theta \\ \quad \\
\text{推导 2 (能量守恒): 速度 } v = L\dot{\theta} \text{, 高度 } h = L(1 - \cos\theta) \\
E = \frac{1}{2} m L^2 \dot{\theta}^2 + m g L (1 - \cos\theta) \\
\frac{dE}{dt} = m L^2 \dot{\theta}\ddot{\theta} + m g L \sin\theta \, \dot{\theta} = 0 \xrightarrow{\text{除以 } m L^2 \dot{\theta}} \ddot{\theta} = -\frac{g}{L}\sin\theta \quad (\text{同一个方程}) \\ \quad \\
\text{这是非线性 ODE (含 } \sin\theta \text{), 没有初等函数的解}
$$

数学基础: [导数的定义与链式法则](../math_basics/calculus.ipynb#derivative-rules) · [常微分方程](../math_basics/calculus.ipynb#ode)

<a id="small-angle"></a>

$$
\text{小角近似 (Small-Angle Approximation): } \sin\theta \text{ 的泰勒展开} \\
\sin\theta = \theta - \frac{\theta^3}{6} + \frac{\theta^5}{120} - \cdots \approx \theta \quad (\theta \text{ 很小时}) \\
\ddot{\theta} = -\frac{g}{L}\theta \implies \text{简谐振动, } \omega_0 = \sqrt{\frac{g}{L}}, \quad T_0 = 2\pi\sqrt{\frac{L}{g}}, \quad \theta(t) = \theta_0 \cos\omega_0 t \\
\text{相对误差: } \frac{\theta - \sin\theta}{\sin\theta} \approx \frac{\theta^2}{6} \text{, 10° (0.1745 rad) 时约 0.5\%} \\
\text{小角近似下周期和振幅无关, 实际上振幅越大周期越长 (sin θ < θ, 回复力更弱)}
$$

数学基础: [泰勒展开](../math_basics/calculus.ipynb#taylor) · 应用: [简谐振动](harmonic_oscillator.ipynb#simple-harmonic)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp, quad
from scipy.special import ellipk

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

g, L = 9.81, 1.0
w0 = np.sqrt(g / L)
T0 = 2 * np.pi / w0

def pendulum(t, s):
    return [s[1], -g / L * np.sin(s[0])]

def linear(t, s):
    return [s[1], -g / L * s[0]]

for deg in [1, 10, 30]:
    th = np.deg2rad(deg)
    print(f'{deg:>2}°: sin = {np.sin(th):.6f}, theta = {th:.6f}, 相对误差 = {(th - np.sin(th)) / np.sin(th):.3e}, theta^2 / 6 = {th**2 / 6:.3e}')

t_eval = np.linspace(0, 4 * T0, 1000)
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
for ax, deg in zip(axs, [10, 90]):
    th0 = np.deg2rad(deg)
    s_nl = solve_ivp(pendulum, [0, t_eval[-1]], [th0, 0], t_eval=t_eval, rtol=1e-10, atol=1e-10)
    s_li = solve_ivp(linear, [0, t_eval[-1]], [th0, 0], t_eval=t_eval, rtol=1e-10, atol=1e-10)
    print(f'theta0 = {deg}°: 线性解 == theta0 cos(w0 t)? {np.allclose(s_li.y[0], th0 * np.cos(w0 * t_eval), atol=1e-7)}, '
          f'非线性和线性的最大差 = {np.max(np.abs(s_nl.y[0] - s_li.y[0])):.4f} rad')
    ax.plot(t_eval, np.degrees(s_nl.y[0]), label='nonlinear sin(theta)')
    ax.plot(t_eval, np.degrees(s_li.y[0]), '--', label='small angle')
    ax.set_title(f'theta0 = {deg} deg')
    ax.set_xlabel('t (s)')
    ax.set_ylabel('theta (deg)')
plot_nicely()
plt.show()

<a id="exact-period"></a>

$$
\text{精确周期: 由能量守恒推导 (完全椭圆积分 Complete Elliptic Integral)} \\ \quad \\
\text{最大摆角 } \theta_0 \text{ 处 } \dot{\theta} = 0 \text{, 两个位置的能量相等:} \\
\frac{1}{2} L^2 \dot{\theta}^2 + g L (1 - \cos\theta) = g L (1 - \cos\theta_0) \implies \dot{\theta} = \sqrt{\frac{2g}{L}(\cos\theta - \cos\theta_0)} \\
dt = \frac{d\theta}{\dot{\theta}} \text{, 从 0 摆到 } \theta_0 \text{ 用 } \frac{T}{4} \text{ 的时间:} \\
T = 4\int_0^{\theta_0} \frac{d\theta}{\sqrt{\frac{2g}{L}(\cos\theta - \cos\theta_0)}} \quad (\text{被积函数在 } \theta_0 \text{ 处趋于无穷, 需要换元}) \\ \quad \\
\text{1. 半角公式 } \cos\theta = 1 - 2\sin^2\frac{\theta}{2} \implies \cos\theta - \cos\theta_0 = 2\left(k^2 - \sin^2\frac{\theta}{2}\right), \quad k = \sin\frac{\theta_0}{2} \\
\text{2. 换元 } \sin\frac{\theta}{2} = k\sin\varphi \ (\theta: 0 \to \theta_0 \text{ 对应 } \varphi: 0 \to \frac{\pi}{2}) \\
\text{两边求导: } \frac{1}{2}\cos\frac{\theta}{2} \, d\theta = k\cos\varphi \, d\varphi \implies d\theta = \frac{2k\cos\varphi \, d\varphi}{\sqrt{1 - k^2\sin^2\varphi}} \\
\sqrt{\cos\theta - \cos\theta_0} = \sqrt{2(k^2 - k^2\sin^2\varphi)} = \sqrt{2} \, k\cos\varphi \\
\text{3. 代入: } T = 4\sqrt{\frac{L}{2g}} \int_0^{\pi/2} \frac{2k\cos\varphi}{\sqrt{1 - k^2\sin^2\varphi} \cdot \sqrt{2} \, k\cos\varphi} \, d\varphi = 4\sqrt{\frac{L}{g}} \underbrace{\int_0^{\pi/2} \frac{d\varphi}{\sqrt{1 - k^2\sin^2\varphi}}}_{K(k)} \\
\text{(} k\cos\varphi \text{ 约掉了, 奇点消失)} \\ \quad \\
T = 4\sqrt{\frac{L}{g}} \, K\!\left(\sin\frac{\theta_0}{2}\right) \quad (\text{注意 scipy.special.ellipk 的参数是 } m = k^2) \\ \quad \\
\text{4. 对小振幅展开: } \frac{1}{\sqrt{1 - x}} = 1 + \frac{x}{2} + \cdots \implies K(k) \approx \int_0^{\pi/2}\left(1 + \frac{k^2}{2}\sin^2\varphi\right) d\varphi = \frac{\pi}{2}\left(1 + \frac{k^2}{4}\right) \\
k^2 = \sin^2\frac{\theta_0}{2} \approx \frac{\theta_0^2}{4} \implies T \approx 2\pi\sqrt{\frac{L}{g}}\left(1 + \frac{\theta_0^2}{16}\right) \\
\theta_0 \to \pi \text{ (倒立) 时 } k \to 1 \text{, } K \to \infty \text{, 周期趋于无穷}
$$

数学基础: [换元积分](../math_basics/calculus.ipynb#integral) · [泰勒展开](../math_basics/calculus.ipynb#taylor) · [数值积分](../math_basics/calculus.ipynb#numerical-integral)

In [ ]:
def period_elliptic(theta0):
    k = np.sin(theta0 / 2)
    return 4 * np.sqrt(L / g) * ellipk(k**2)

def period_quad(theta0):
    # 换元后的积分, 手动用 quad
    k = np.sin(theta0 / 2)
    K, _ = quad(lambda phi: 1 / np.sqrt(1 - k**2 * np.sin(phi)**2), 0, np.pi / 2)
    return 4 * np.sqrt(L / g) * K

def period_simulation(theta0):
    # 从 theta0 静止释放, theta 第一次向下穿过 0 是 1/4 周期
    def cross(t, s):
        return s[0]
    cross.terminal = True
    cross.direction = -1
    sol = solve_ivp(pendulum, [0, 100], [theta0, 0], events=cross, rtol=1e-12, atol=1e-12)
    return 4 * sol.t_events[0][0]

def period_original_integral(theta0):
    # 没换元的原积分, 端点 theta0 处有 1/sqrt(theta0 - theta) 的奇点
    # 把奇点因子交给 quad 的 alg 权重: 被积函数 = w(theta) * (theta0 - theta)^(-1/2)
    # 和差化积 cos(th) - cos(theta0) = 2 sin((theta0 + th)/2) sin((theta0 - th)/2), 避免端点处 0/0
    # sin(x/2) / x = 0.5 * np.sinc(x / (2 pi)), np.sinc(u) = sin(pi u) / (pi u)
    ratio = lambda th: np.sin((theta0 + th) / 2) * np.sinc((theta0 - th) / (2 * np.pi))   # (cos th - cos theta0) / (theta0 - th)
    w = lambda th: 1 / np.sqrt(2 * g / L * ratio(th))
    val, _ = quad(w, 0, theta0, weight='alg', wvar=(0, -0.5))
    return 4 * val

print(f'小角近似周期 T0 = {T0:.6f}')
for deg in [5, 30, 60, 90, 120, 170]:
    th0 = np.deg2rad(deg)
    print(f'theta0 = {deg:>3}°: 椭圆积分 = {period_elliptic(th0):.6f}, 换元后 quad = {period_quad(th0):.6f}, '
          f'原积分 = {period_original_integral(th0):.6f}, 模拟 = {period_simulation(th0):.6f}, 级数近似 = {T0 * (1 + th0**2 / 16):.6f}')

degs = np.linspace(1, 175, 200)
plt.figure(figsize=(4, 3))
plt.plot(degs, [period_elliptic(np.deg2rad(d)) / T0 for d in degs], label='exact 4 sqrt(L/g) K')
plt.plot(degs, 1 + np.deg2rad(degs)**2 / 16, '--', label='1 + theta0^2 / 16')
plt.axhline(1, color='gray', linestyle=':', label='small angle')
plt.xlabel('theta0 (deg)')
plt.ylabel('T / T0')
plot_nicely()
plt.show()

<a id="phase-portrait"></a>

$$
\text{相图 (Phase Portrait): 以 } (\theta, \omega = \dot{\theta}) \text{ 为坐标, 每条轨迹是一条能量等高线} \\
\frac{E}{m L^2} = \frac{1}{2}\omega^2 - \frac{g}{L}\cos\theta + \frac{g}{L} = \text{常数} \quad (\text{能量守恒, 轨迹不会相交}) \\ \quad \\
\text{1. 能量低: 来回摆动 (libration), 相图上是闭合曲线, 小振幅时是椭圆 (简谐振动)} \\
\text{2. 能量高: 越过最高点一直转圈 (rotation), 相图上是不闭合的波浪线} \\
\text{3. 分界线 (Separatrix): 刚好能到达倒立位置 } \theta = \pi \text{ 且速度为 0, } E = 2 m g L \\
\frac{1}{2}\omega^2 = \frac{g}{L}(1 + \cos\theta) = \frac{2g}{L}\cos^2\frac{\theta}{2} \implies \omega = \pm 2\sqrt{\frac{g}{L}}\cos\frac{\theta}{2} \\ \quad \\
\text{平衡点: } \dot{\theta} = 0, \ \sin\theta = 0 \\
\theta = 0 \text{: 稳定 (中心 center), 附近线性化 } \ddot{\theta} = -\frac{g}{L}\theta \\
\theta = \pi \text{: 不稳定 (鞍点 saddle), 令 } \theta = \pi + \epsilon \text{, } \sin(\pi + \epsilon) = -\sin\epsilon \approx -\epsilon \implies \ddot{\epsilon} = +\frac{g}{L}\epsilon \text{, 指数增长} \\
\text{(和 Hessian 判断极值一样: 势能 } U = -\cos\theta \text{ 在 0 处是极小值, 在 } \pi \text{ 处是极大值)}
$$

数学基础: [极值与 Hessian](../math_basics/calculus.ipynb#extremum) · [高阶 ODE 转一阶方程组](../math_basics/calculus.ipynb#ode-system)

In [ ]:
energy = lambda th, om: 0.5 * om**2 - g / L * np.cos(th) + g / L

TH, OM = np.meshgrid(np.linspace(-2 * np.pi, 2 * np.pi, 400), np.linspace(-9, 9, 400))
th_line = np.linspace(-2 * np.pi, 2 * np.pi, 400)
sep = 2 * np.sqrt(g / L) * np.abs(np.cos(th_line / 2))

plt.figure(figsize=(8, 3))
plt.contour(TH, OM, energy(TH, OM), levels=20, colors='gray', linewidths=0.6, alpha=0.6)
plt.plot(th_line, sep, 'r', label='separatrix')
plt.plot(th_line, -sep, 'r')

t_eval = np.linspace(0, 6, 3000)
for th0, om0, name in [(np.deg2rad(60), 0, 'libration'), (0, 6.5, 'rotation')]:
    sol = solve_ivp(pendulum, [0, 6], [th0, om0], t_eval=t_eval, rtol=1e-10, atol=1e-10)
    E_track = energy(sol.y[0], sol.y[1])
    print(f'{name:<9}: 能量变化 = {E_track.max() - E_track.min():.2e}, E = {E_track[0]:.4f}, 分界线能量 2g/L = {2 * g / L:.4f}')
    th_wrapped = (sol.y[0] + np.pi) % (2 * np.pi) - np.pi
    plt.plot(np.where(np.abs(np.diff(th_wrapped, prepend=th_wrapped[0])) > np.pi, np.nan, th_wrapped), sol.y[1], label=name)
plt.plot([0], [0], 'go', label='stable (0, 0)')
plt.plot([-np.pi, np.pi], [0, 0], 'kx', markersize=8, label='unstable (+-pi, 0)')
plt.xlabel('theta')
plt.ylabel('omega')
plt.xlim(-2 * np.pi, 2 * np.pi)
plot_nicely()
plt.show()

# 倒立附近: 微小扰动指数增长, 增长率 sqrt(g / L)
eps0 = 1e-6
sol = solve_ivp(pendulum, [0, 3], [np.pi + eps0, 0], t_eval=[1, 2, 3], rtol=1e-12, atol=1e-14)
print(f'倒立 + 1e-6 扰动: 1, 2, 3 秒后偏离 = {np.round(sol.y[0] - np.pi, 8)}, 线性化 eps0 cosh(sqrt(g/L) t) = {np.round(eps0 * np.cosh(w0 * np.array([1, 2, 3])), 8)}')